<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0; font-size: 2em;">⚙️ Lab 01: Environment Setup & Validation</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">
    Contoso Travel Workshop — Microsoft Foundry Agent Observability
  </p>
</div>

## 🎯 私たちが学ぶこと

このラボでは、開発環境を**Microsoft Foundry**と**Azure AI Projects SDK**と連携するように構成します。これは、**Contoso Travel**エージェントシステム（顧客がフライト、ホテル、レンタカーを探すのを支援するマルチエージェント型旅行アシスタント）を構築するための基盤となります。

---


## 🧳 Contoso Travel のストーリー

**Contoso Travel** は、人間のアドバイザーでは対応しきれないほどの顧客からの問い合わせに対応するため、AI を活用した旅行アシスタントを開発しています。そのビジョンは、フライト、ホテル、レンタカーそれぞれを担当するインテリジェントなエージェントが連携して旅行プラン全体を作成するシステムです。**あなたは、このシステムを開発する開発者です。** Foundry プロジェクト (Lab 00) の設定が完了しました。Azure リソースはプロビジョニングされ、モデルはデプロイされ、Application Insights も準備完了です。

### 🔴 現在直面している問題
- クラウドプロジェクトはありますが、**ローカル開発環境**はまだ接続されていません。
- 適切な SDK パッケージをインストールし、資格情報を設定し、コードが実際に Foundry サービスと通信できることを確認する必要があります。
- この接続が壊れていると、このラボ以降のすべてのラボが静かに失敗します — エージェントの構築中に認証問題をデバッグするのは非常に困難です。

### ✅ このラボで解決すること

このラボでは、開発環境を本番準備状態にします：
 - 依存関係のインストール、
 - 環境変数の設定、
 - Azure 接続の検証、
 - OpenAI クライアントのテスト、
 - Contoso Travel のサンプルデータのプレビュー。
ラボの最後には、以降のラボでコードを実行する際に「そのまま動作する」状態になります。

## 1. 依存関係のインストール

Azure AI Projects SDK、認証ライブラリ、トレーシング用の OpenTelemetry、旅行データを扱うための pandas が必要です。

---


In [ ]:
# azure-ai-projects: Foundry SDK for agents, evals, telemetry
# azure-identity: DefaultAzureCredential auth chain
# opentelemetry + azure-monitor: distributed tracing (Labs 04+)
%pip install azure-ai-projects>=2.0.0 azure-identity python-dotenv opentelemetry-sdk azure-core-tracing-opentelemetry azure-monitor-opentelemetry pandas --quiet

## 2. 環境変数の設定

`.env` ファイルは Lab 00 の間に `labs/notebooks/` に作成されました。値が Foundry ポータルと一致していることを確認してください：

| 変数 | 場所 |
|----------|--------------|
| `AZURE_AI_PROJECT_ENDPOINT` | Foundry portal → Project Overview |
| `AZURE_AI_MODEL_DEPLOYMENT_NAME` | Foundry portal → Models + Endpoints → Name column |
| `MODEL_ENDPOINT` | Models + endpoints → click deployment → Target URI (base URL only) |
| `MODEL_API_KEY` | Models + endpoints → click deployment → Show key |

---


In [ ]:
# Load and validate environment variables from shared .env
import os
from dotenv import load_dotenv

# abspath(".") = notebook CWD (e.g. labs/notebooks/1-prompt-agents/)
# dirname() goes up one level → labs/notebooks/ where .env lives.
# This pattern lets all lab subdirs share a single .env file.
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

required_vars = [
    "AZURE_AI_PROJECT_ENDPOINT",
    "AZURE_AI_MODEL_DEPLOYMENT_NAME",
]

# Optional vars only needed for Lab 06 (Red Teaming)
optional_vars = [
    "MODEL_ENDPOINT",
    "MODEL_API_KEY",
]

print("✅ Checking required environment variables...\n")
all_set = True
for var in required_vars:
    value = os.environ.get(var)
    if value:
        # Truncate to avoid exposing full secrets in output
        display = value[:12] + "..." + value[-4:] if len(value) > 20 else value
        print(f"  ✅ {var} = {display}")
    else:
        print(f"  ❌ {var} is NOT SET")
        all_set = False

print("\n📋 Checking optional environment variables...\n")
for var in optional_vars:
    value = os.environ.get(var)
    if value:
        display = value[:12] + "..." + value[-4:] if len(value) > 20 else value
        print(f"  ✅ {var} = {display}")
    else:
        print(f"  ⚠️  {var} is not set (needed for Red Teaming lab)")

if all_set:
    print("\n🎉 All required variables are configured!")
else:
    print("\n⚠️  Please set the missing variables in your .env file before continuing.")

## 3. Azure への接続の検証

SDK を使用して、Microsoft Foundry プロジェクトに接続できることを確認しましょう。

---


In [ ]:
# Validate SDK can reach the Foundry project endpoint
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]

# Chains through az CLI, managed identity, env vars automatically
credential = DefaultAzureCredential()
# Central client for all Foundry features: agents, evals, telemetry
project_client = AIProjectClient(endpoint=endpoint, credential=credential)

print(f"✅ Connected to Microsoft Foundry!")
print(f"   Endpoint: {endpoint}")

## 4. OpenAI クライアントの検証

Azure AI Projects SDK は OpenAI 互換のクライアントを提供します。簡単なテストプロンプトで動作を確認しましょう。

---


In [ ]:
# get_openai_client() returns a fully-configured OpenAI client
# that inherits the project endpoint and credential — no manual setup
openai_client = project_client.get_openai_client()

model_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

# Uses Chat Completions API (standard across Azure OpenAI)
response = openai_client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "user", "content": "Say 'Hello from Contoso Travel!' in one sentence."}
    ]
)

print(f"✅ Model '{model_name}' is responding!")
print(f"   Response: {response.choices[0].message.content}")

## 5. Contoso Travel サンプルデータの確認

私たちの旅行エージェントは、フライト、ホテル、レンタカーの CSV データファイルを使用します。エージェントが扱うデータをプレビューしてみましょう。

---


In [ ]:
# Load and preview Contoso Travel flight inventory
import pandas as pd

# Relative to notebook dir → ../../data/ reaches labs/data/
data_path = "../../data/contoso-travel"

flights = pd.read_csv(f"{data_path}/flights.csv")
print(f"✈️  Flights: {len(flights)} records")
print(f"   Routes: {flights['origin'].nunique()} origins → {flights['destination'].nunique()} destinations")
print(f"   Price range: ${flights['price_usd'].min():.0f} - ${flights['price_usd'].max():.0f}")
flights.head(3)

In [ ]:
# Preview hotel inventory across destination cities
hotels = pd.read_csv(f"{data_path}/hotels.csv")
print(f"🏨 Hotels: {len(hotels)} records")
print(f"   Cities: {', '.join(hotels['city'].unique())}")
print(f"   Star ratings: {hotels['star_rating'].min()}-{hotels['star_rating'].max()} stars")
print(f"   Price range: ${hotels['price_per_night_usd'].min():.0f} - ${hotels['price_per_night_usd'].max():.0f}/night")
hotels.head(3)

In [ ]:
# Preview car rental options by city and vehicle type
cars = pd.read_csv(f"{data_path}/car_rentals.csv")
print(f"🚗 Car Rentals: {len(cars)} records")
print(f"   Cities: {', '.join(cars['city'].unique())}")
print(f"   Types: {', '.join(cars['car_type'].unique())}")
print(f"   Price range: ${cars['price_per_day_usd'].min():.0f} - ${cars['price_per_day_usd'].max():.0f}/day")
cars.head(3)

## 6. まとめと次のステップ

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #2e8b57; padding: 18px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <h3 style="margin-top: 0; color: #1a1a1a;">✅ 達成したこと</h3>
  <ul style="margin-bottom: 0;">
  <li>必要な SDK 依存関係をすべてインストールしました</li>
  <li>Microsoft Foundry 用の環境変数を設定しました</li>
  <li>Foundry プロジェクトへの接続を検証しました</li>
  <li>Contoso Travel のサンプルデータ（フライト、ホテル、レンタカー）を確認しました</li>
  </ul>
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #ff9800; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <strong>➡️ 次のステップ:</strong> <a href="lab-02-agent.ipynb">Lab 02</a> では、最初のプロンプトエージェント — Contoso Travel コンシェルジュを作成します！
</div>

---


In [ ]:
# Release HTTP connections and token caches to avoid leaks
# Order matters: close leaf clients before the parent project client
openai_client.close()
project_client.close()
credential.close()
print("✅ Clients closed. Setup complete!")

---